# TA - Selenium
---

Es el uso de software para testear otro software y comparar el resultado obtenido con el esperado

## ¿Por qué automatizar pruebas?

* Evita tareas repetitivas
* Reducción de errores
* Incrementar la cantidad de test a realizar
* Es importante para CI/CD
* Reduce costos
* Permite que los Testers se enfoquen en pensar escenarios nuevos y conocer más el negocio

## HTML

* Lenguaje de marcado que se utiliza para el desarrollo de páginas web
* Permite estructurar el texto de los documentos de forma jerárquica
*  Se basa en etiquetas que determinan cómo debe mostrarse el texto y los atributos que pueden tomar dichas etiquetas

```
<etiqueta atributo="valor">Contenido</etiqueta>
````

## XPATH

* Lenguaje que se usa para buscar la ubicación de cualquier elemento en una página web utilizando la estructura HTML

| Tipos        | Ejemplos                                                  |
| --           | --                                                        |
| **absoluto** | /html/body/main/div/div/div/div/div[1]/div/form/div/input |
| **relativo** | //*[@id="F1:username"]                                    |

## Selenium

* Escribir pruebas funcionales y de aceptación

## Instalación

```
pip install selenium
```

## Ejemplo

In [13]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get('http://www.python.org')
assert 'Python' in driver.title
elem = driver.find_element(By.NAME, 'q')
elem.clear()
elem.send_keys('pycon')
elem.send_keys(Keys.RETURN)
assert 'No results found.' not in driver.page_source
driver.close()

## Estrategias de selección de elementos web (By)

```HTML
<input id="F1:username" name="F1:username" class="form-control" maxlength="11" size="11" type="number">
```
* driver.find_element(By.ID, "F1:username")
* driver.find_element(By.NAME, "F1:username")
* driver.find_element(By.XPATH, "//*[@id='F1:username']")
* driver.find_element(By.CLASS_NAME,"form-control")
* driver.find_element(By.TAG_NAME,"input")
* driver.find_element(By.CSS_SELECTOR,“#F1:username”)

```HTML
<a href="https://www.afip.gob.ar/claveFiscal/recupero.asp" class="text-black" target="_blank">¿Olvidaste tu clave?</a>
```

* driver.find_element(By.LINK_TEXT,"¿Olvidaste tu clave?")
* driver.find_element(By.PARTIAL_LINK_TEXT,"¿Olvidaste")

## Esperas

* time.sleep(n): no es aconsejable
* driver.implicitly_wait(10)
* esperas explícitas:

```py
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)
element = wait.until(EC.element_to_be_clickable((By.ID, 'someid')))
```

## Ejemplo con unittest

In [12]:
import unittest
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

class PythonOrgSearch(unittest.TestCase):

    def setUp(self):
        self.driver = webdriver.Chrome()

    def test_search_in_python_org(self):
        driver = self.driver
        driver.get("http://www.python.org")
        self.assertIn("Python", driver.title)
        elem = driver.find_element(By.NAME, "q")
        elem.send_keys("pycon")
        elem.send_keys(Keys.RETURN)
        self.assertNotIn("No results found.", driver.page_source)

    def tearDown(self):
        self.driver.close()

if __name__ == "__main__":
    # ejecución como archivo: unittest.main()
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 4.167s

OK


## Ejemplo con pytest

```
pip install pytest
```

In [10]:
%%file test_pytest.py
import pytest

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

@pytest.mark.usefixtures('class_setup')
class TestPythonOrgSearch:

    @pytest.fixture
    def class_setup(self):
        self.driver = webdriver.Chrome()
        yield
        self.driver.close()

    def test_search_in_python_org(self):
        self.driver.get('http://www.python.org')
        assert 'Python' in self.driver.title
        elem = self.driver.find_element(By.NAME, 'q')
        elem.clear()
        elem.send_keys('pycon')
        elem.send_keys(Keys.RETURN)
        assert 'No results found.' not in self.driver.page_source

Writing test_pytest.py


In [11]:
!pytest test_pytest.py

============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0
rootdir: /Users/ppando/Materias/pweb/doc
plugins: Faker-37.3.0, mock-3.14.1, typeguard-4.4.4, anyio-4.2.0
collected 1 item                                                               

test_pytest.py .                                                         [100%]

============================== 1 passed in 17.64s ==============================


## Externalización de datos

```
pip install pyaml_env
```

In [18]:
%%file test_pytest_yaml.py
import pytest

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from pyaml_env import parse_config

@pytest.mark.usefixtures('class_setup')
class TestPythonOrgSearch:

    datos = parse_config('data.yaml')

    @pytest.fixture
    def class_setup(self):
        self.driver = webdriver.Chrome()
        yield
        self.driver.close()

    def test_search_in_python_org(self):
        self.driver.get(self.datos['url'])
        assert 'Python' in self.driver.title
        elem = self.driver.find_element(By.NAME, 'q')
        elem.clear()
        elem.send_keys('pycon')
        elem.send_keys(Keys.RETURN)
        assert self.datos['expected'] not in self.driver.page_source

Overwriting test_pytest_yaml.py


In [19]:
!pytest test_pytest_yaml.py

============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0
rootdir: /Users/ppando/Materias/pweb/doc
plugins: Faker-37.3.0, mock-3.14.1, typeguard-4.4.4, anyio-4.2.0
collected 1 item                                                               

test_pytest_yaml.py .                                                    [100%]

============================== 1 passed in 3.55s ===============================


## Parametrización

In [20]:
%%file test_pytest_yaml_param.py
import pytest

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from pyaml_env import parse_config

@pytest.mark.usefixtures('class_setup')
class TestPythonOrgSearch:

    datos = parse_config('data.yaml')

    @pytest.fixture
    def class_setup(self):
        self.driver = webdriver.Chrome()
        yield
        self.driver.close()
        
    @pytest.mark.parametrize('filtros', datos['search'].values())
    def test_search_in_python_org(self, filtros):
        self.driver.get(self.datos['url'])
        assert 'Python' in self.driver.title
        elem = self.driver.find_element(By.NAME, 'q')
        elem.clear()
        elem.send_keys(filtros['filtro'])
        elem.send_keys(Keys.RETURN)
        assert self.datos['expected'] not in self.driver.page_source

Writing test_pytest_yaml_param.py


In [21]:
!pytest test_pytest_yaml_param.py

============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0
rootdir: /Users/ppando/Materias/pweb/doc
plugins: Faker-37.3.0, mock-3.14.1, typeguard-4.4.4, anyio-4.2.0
collected 2 items                                                              

test_pytest_yaml_param.py ..                                             [100%]

============================== 2 passed in 7.74s ===============================


## Ejercicio

* Testear en el sitio web de la UNO, la duración de la carrera: Tecnicatura Universitaria en Tecnologías Web